# Getrennte Hinweise für Abbruch und Nichtbestehen

Fragestellung: Lassen sich am Ende von Tag 28 neben dem gemeinsamen Risiko
auch die späteren Ergebnisse „Withdrawn“ und „Fail“ unterscheiden?

Drei Zielklassen: Abbruch (Withdrawn), Nichtbestehen (Fail) und
erfolgreicher Abschluss (Pass oder Distinction).

Wir verwenden dieselben Tag-28-Merkmale und prüfen zeitlich:
Training auf Kurspräsentationen von 2013, Prüfung auf 2014J.
Personen aus der Prüfung werden aus dem Training ausgeschlossen.

In [1]:
from pathlib import Path

import pandas as pd

# Den Projektordner finden, egal ob das Notebook direkt dort
# oder im Unterordner "notebooks" liegt.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

schluessel = ["code_module", "code_presentation", "id_student"]

# Die bereits fertige Tag-28-Modellbasis einlesen.
basis = pd.read_csv(
    projektordner / "data" / "processed" / "oulad_tag28_modellbasis.csv"
)

# Das endgültige Kursergebnis dient ausschließlich als Zielvariable.
# Es wird später nicht als Eingabemerkmal an das Modell übergeben.
info = pd.read_csv(
    projektordner / "Daten OULAD" / "studentInfo.csv",
    usecols=schluessel + ["final_result"],
)

assert not basis.duplicated(schluessel).any()
assert not info.duplicated(schluessel).any()

daten = basis.merge(
    info,
    on=schluessel,
    how="left",
    validate="one_to_one",
)

zuordnung = {
    "Withdrawn": "Abbruch",
    "Fail": "Nichtbestehen",
    "Pass": "Erfolgreich",
    "Distinction": "Erfolgreich",
}
daten["ziel_3klassen"] = daten["final_result"].map(zuordnung)

# Prüfen, ob alle Fälle zugeordnet wurden und ob die neue Zielvariable
# mit unserer bisherigen gemeinsamen Zielvariable übereinstimmt.
assert daten["ziel_3klassen"].notna().all()
assert (
    daten["abbruch_oder_fail"]
    == daten["ziel_3klassen"].ne("Erfolgreich")
).all()

print("Kurseinträge:", len(daten))
display(daten["ziel_3klassen"].value_counts())

Kurseinträge: 27422


ziel_3klassen
Erfolgreich      15378
Nichtbestehen     7032
Abbruch           5012
Name: count, dtype: int64

In [2]:
# Auf 2013 trainieren und auf der späteren Präsentation 2014J prüfen.
training = daten[
    daten["code_presentation"].isin(["2013B", "2013J"])
].copy()

pruefung = daten[
    daten["code_presentation"].eq("2014J")
].copy()

# Personen, die auch im Prüfjahr vorkommen, aus dem Training entfernen.
gemeinsame_personen = set(training["id_student"]) & set(pruefung["id_student"])
training = training[
    ~training["id_student"].isin(gemeinsame_personen)
].copy()

assert set(training["id_student"]).isdisjoint(pruefung["id_student"])

verteilung = pd.DataFrame({
    "Training": training["ziel_3klassen"].value_counts(),
    "Prüfung 2014J": pruefung["ziel_3klassen"].value_counts(),
}).fillna(0).astype(int)

print("Wegen Überschneidung ausgeschlossene Personen:", len(gemeinsame_personen))
print("Training:", len(training), "| Prüfung:", len(pruefung))
display(verteilung)

Wegen Überschneidung ausgeschlossene Personen: 756
Training: 11031 | Prüfung: 9123


,Training,Prüfung 2014J
ziel_3klassen,,
Erfolgreich,6150,5455
Nichtbestehen,3139,1975
Abbruch,1742,1693


In [3]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

# Genau die Merkmale aus dem bisherigen Tag-28-Modell.
kategorien = ["code_module", "code_presentation"]
zahlen = [
    "anmeldetag",
    "klicks_tag_0_bis_6",
    "klicks_tag_7_bis_14",
    "klicks_tag_15_bis_28",
    "assessments_faellig_bis28",
    "abgaben_bis28",
    "assessments_fehlend_bis28",
    "banked_faellige_bis28",
]
alle_merkmale = kategorien + zahlen

# Einfache Referenz: nur Kurs und Anmeldetag.
referenz_merkmale = kategorien + ["anmeldetag"]


def erstelle_modell(merkmale, klassifikator):
    """Bereitet Kursangaben und Zahlen wie im bisherigen Modellvergleich auf."""
    zahlenspalten = [
        spalte for spalte in merkmale if spalte not in kategorien
    ]

    vorbereitung = ColumnTransformer(
        transformers=[
            (
                "kategorien",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                kategorien,
            ),
            ("zahlen", StandardScaler(), zahlenspalten),
        ]
    )

    return Pipeline([
        ("vorbereitung", vorbereitung),
        ("klassifikator", klassifikator),
    ])


# Drei Klassen: Abbruch, Nichtbestehen, erfolgreicher Abschluss.
modell_drei = erstelle_modell(
    alle_merkmale,
    HistGradientBoostingClassifier(random_state=42),
)
modell_drei.fit(training[alle_merkmale], training["ziel_3klassen"])

# Referenz für die Frage, ob die Aktivitäts- und Assessmentdaten helfen.
referenz_drei = erstelle_modell(
    referenz_merkmale,
    LogisticRegression(max_iter=1000),
)
referenz_drei.fit(
    training[referenz_merkmale],
    training["ziel_3klassen"],
)

# Bisherige gemeinsame Fragestellung als direkter Vergleich
# auf exakt demselben Training und Prüfteil.
modell_gemeinsam = erstelle_modell(
    alle_merkmale,
    HistGradientBoostingClassifier(random_state=42),
)
modell_gemeinsam.fit(
    training[alle_merkmale],
    training["abbruch_oder_fail"].astype(int),
)

# Vorhersagewerte für 2014J; Spaltennamen kommen direkt aus dem Modell.
werte_drei = pd.DataFrame(
    modell_drei.predict_proba(pruefung[alle_merkmale]),
    columns=modell_drei.named_steps["klassifikator"].classes_,
    index=pruefung.index,
)
werte_referenz = pd.DataFrame(
    referenz_drei.predict_proba(pruefung[referenz_merkmale]),
    columns=referenz_drei.named_steps["klassifikator"].classes_,
    index=pruefung.index,
)
werte_gemeinsam = modell_gemeinsam.predict_proba(
    pruefung[alle_merkmale]
)[:, 1]

print("Klassen des neuen Modells:", list(werte_drei.columns))
print("Vorhersagen für Kurseinträge:", len(werte_drei))

Klassen des neuen Modells: ['Abbruch', 'Erfolgreich', 'Nichtbestehen']
Vorhersagen für Kurseinträge: 9123


In [5]:
from sklearn.metrics import average_precision_score

ergebnisse = []

for klasse in ["Abbruch", "Nichtbestehen"]:
    # Für jede Klasse gilt: Gehört der Fall zu dieser Klasse oder nicht?
    wahr = pruefung["ziel_3klassen"].eq(klasse).astype(int)

    ergebnisse.append({
        "Ergebnis": klasse,
        "Anteil im Prüfteil": wahr.mean(),
        "AP nur Kurs + Anmeldung": average_precision_score(
            wahr, werte_referenz[klasse]
        ),
        "AP alle Merkmale": average_precision_score(
            wahr, werte_drei[klasse]
        ),
    })

display(pd.DataFrame(ergebnisse).round(3))

# Zusätzlich prüfen wir, ob die Summe der beiden Risikowerte
# als gemeinsamer Hinweis mit dem bisherigen Binärmodell mithält.
wahr_gemeinsam = pruefung["abbruch_oder_fail"].astype(int)
wert_drei_gemeinsam = (
    werte_drei["Abbruch"] + werte_drei["Nichtbestehen"]
)

print(
    "AP gemeinsamer Hinweis – bisheriges Binärmodell:",
    round(average_precision_score(wahr_gemeinsam, werte_gemeinsam), 3),
)
print(
    "AP gemeinsamer Hinweis – Drei-Klassen-Modell:",
    round(average_precision_score(wahr_gemeinsam, wert_drei_gemeinsam), 3),
)
print(
    "Anteil Abbruch oder Nichtbestehen:",
    round(wahr_gemeinsam.mean(), 3),
)

,Ergebnis,Anteil im Prüfteil,AP nur Kurs + Anmeldung,AP alle Merkmale
0,Abbruch,0.186,0.190,0.269
1,Nichtbestehen,0.216,0.245,0.374


AP gemeinsamer Hinweis – bisheriges Binärmodell: 0.635
AP gemeinsamer Hinweis – Drei-Klassen-Modell: 0.633
Anteil Abbruch oder Nichtbestehen: 0.402


In [6]:
import numpy as np

# Eine Auswertungstabelle mit den tatsächlich später eingetretenen
# Ergebnissen und den vorhergesagten Werten erstellen.
auswertung = pruefung[
    schluessel + ["ziel_3klassen"]
].copy()

auswertung["risiko_gesamt"] = (
    werte_drei["Abbruch"] + werte_drei["Nichtbestehen"]
).to_numpy()

auswertung["richtung"] = np.where(
    werte_drei["Abbruch"].to_numpy()
    >= werte_drei["Nichtbestehen"].to_numpy(),
    "Abbruch",
    "Nichtbestehen",
)

# Innerhalb jeder Kurspräsentation die höchsten gemeinsamen
# Risikowerte zuerst anordnen. Die Position dient nur der Auswahl.
kurs_spalten = ["code_module", "code_presentation"]

auswertung = auswertung.sort_values(
    kurs_spalten + ["risiko_gesamt"],
    ascending=[True, True, False],
    kind="stable",
).copy()

auswertung["position"] = (
    auswertung.groupby(kurs_spalten).cumcount() + 1
)
auswertung["kursgroesse"] = (
    auswertung.groupby(kurs_spalten)["id_student"]
    .transform("size")
)

zusammenfassung = []
klassen_ergebnisse = []

for anteil in (10, 20):
    # Pro Kurspräsentation die obersten 10 % bzw. 20 % auswählen.
    anzahl_je_kurs = np.ceil(
        auswertung["kursgroesse"] * anteil / 100
    ).astype(int)

    hinweise = auswertung.loc[
        auswertung["position"] <= anzahl_je_kurs
    ].copy()

    tatsaechlicher_fall = hinweise["ziel_3klassen"].ne(
        "Erfolgreich"
    )
    richtige_richtung = (
        hinweise["richtung"] == hinweise["ziel_3klassen"]
    )

    treffer = int(tatsaechlicher_fall.sum())
    richtige_art = int(richtige_richtung.sum())

    # Einfache Vergleichsregel: Bei jedem Hinweis
    # "Nichtbestehen" angeben. Das war im Training
    # die häufigere der beiden Risikoklassen.
    baseline_richtige_art = int(
        hinweise["ziel_3klassen"].eq("Nichtbestehen").sum()
    )

    zusammenfassung.append({
        "Hinweisanteil je Kurs": f"{anteil} %",
        "Hinweise": len(hinweise),
        "Abbruch oder Fail": treffer,
        "Trefferquote": treffer / len(hinweise),
        "Richtige Art unter Treffern": richtige_art / treffer,
        "Vergleich: immer Nichtbestehen": (
            baseline_richtige_art / treffer
        ),
    })

    for klasse in ["Abbruch", "Nichtbestehen"]:
        korrekt = int(
            (
                hinweise["ziel_3klassen"].eq(klasse)
                & hinweise["richtung"].eq(klasse)
            ).sum()
        )
        alle_faelle = int(
            pruefung["ziel_3klassen"].eq(klasse).sum()
        )

        klassen_ergebnisse.append({
            "Hinweisanteil je Kurs": f"{anteil} %",
            "Ergebnis": klasse,
            "Korrekt erkannte Fälle": korrekt,
            "Alle Fälle im Prüfteil": alle_faelle,
            "Recall mit richtiger Art": korrekt / alle_faelle,
        })

    if anteil == 10:
        print("Tatsächliches Ergebnis × vorgeschlagene Richtung bei 10 %:")
        display(pd.crosstab(
            hinweise["ziel_3klassen"],
            hinweise["richtung"],
        ))

print("Qualität der Hinweise und ihrer Richtung:")
display(pd.DataFrame(zusammenfassung).round(3))

print("Wie viele Fälle jeder Art finden wir mit richtiger Richtung?")
display(pd.DataFrame(klassen_ergebnisse).round(3))

Tatsächliches Ergebnis × vorgeschlagene Richtung bei 10 %:


richtung,Abbruch,Nichtbestehen
ziel_3klassen,,
Abbruch,31,278
Erfolgreich,19,176
Nichtbestehen,30,381


Qualität der Hinweise und ihrer Richtung:


,Hinweisanteil je Kurs,Hinweise,Abbruch oder Fail,Trefferquote,Richtige Art unter Treffern,Vergleich: immer Nichtbestehen
0,10 %,915,720,0.787,0.572,0.571
1,20 %,1827,1218,0.667,0.583,0.580


Wie viele Fälle jeder Art finden wir mit richtiger Richtung?


,Hinweisanteil je Kurs,Ergebnis,Korrekt erkannte Fälle,Alle Fälle im Prüfteil,Recall mit richtiger Art
0,10 %,Abbruch,31,1693,0.018
1,10 %,Nichtbestehen,381,1975,0.193
2,20 %,Abbruch,48,1693,0.028
3,20 %,Nichtbestehen,662,1975,0.335


## Ergebnis und Entscheidung

Das Drei-Klassen-Modell erreicht für den gemeinsamen Hinweis eine AP von 0,633
gegenüber 0,635 beim bisherigen Binärmodell. Die getrennten Rangfolgen enthalten
Signale für Abbruch (AP 0,269; Anteil 0,186) und Nichtbestehen
(AP 0,374; Anteil 0,216).

Für eine Empfehlung der Art des Unterstützungsbedarfs reicht die Leistung
derzeit nicht: Bei 10 % Hinweisen je Kurs stimmt die vorgeschlagene Richtung
bei 57,2 % der tatsächlichen Fälle. Die einfache Vergleichsregel
„immer Nichtbestehen“ erreicht 57,1 %. Von 309 ausgewählten Personen mit
späterem Abbruch erhalten nur 31 die Richtung „Abbruch“.

Der Prototyp verwendet daher weiterhin den gemeinsamen Hinweis
„Abbruch oder Nichtbestehen“ und zeigt bekannte Aktivitäts- und
Assessmentmerkmale als Gesprächsanlass. Er gibt keine individuelle Ursache
oder passende Intervention als Modellvorhersage aus.

Die Prüfung auf 2014J ist explorativ: Diese Kurspräsentation wurde bereits
bei früheren Analysen betrachtet und stellt für diese neue Entscheidung
keinen unangetasteten Prüfdatensatz dar.